### Error Handling with Try-Except

1. Graceful error handloing with Try-Except
When an exception occurs, it its not handled, the program will terminate abruptly (crush) and print a traceback.
Examples include ZeroDivisionError, TypeError, FileNotFoundError, IndexError etc.

2. Try Except Block

syntax:

try:
    <!-- code to run -->
except ExceptionType:
    <!-- code to run if Exception type occues in the try block -->
    


In [9]:
# example Zero Divion error

def safe_divide(numerator, denominator):
    try:
        result = numerator / denominator
        print(f"Result of Divisio: {result}")
    except ZeroDivisionError as ze:
        print(f"Error: Can not divide by Zero: {ze}")
    except TypeError as te:
        print(f"Error: Both inputs must be numbers: {te}")
    except Exception as e:
        print(f"Unexpected error occured: {e}")

safe_divide("kmm","0")
    


Error: Both inputs must be numbers: unsupported operand type(s) for /: 'str' and 'str'


##### Preventing Crashes in Pipelines

Crucial in automated pipelines (e.g data processing, web-scrapping, batch-jobs) - you don't want one step failure tp affect whole pipeline.

The Else Block:
The code in the else block only executes if the try block completes without raising any exception
Usefull for code that should only run if everything was successfull.

The finally block
An optional block included after try except, else.
This code will always execute regardless if exception was raised or not.
This is essesntial for cleanup opersations like closing files, database connections to free resources


Syntax

try:
    # Code that might raise an exception
    
except SpecificErrorType:
    # Handle specific error
    
else:
    # Code to execute if NO exception occurred in the 'try' block
    
finally:
    # Code that ALWAYS executes (cleanup, logging, etc.)
    

In [11]:
operations = [
    (10, 2, "divide"),
    (5, 0, "divide"),   # Will cause ZeroDivisionError
    ("hello", 2, "multiply"), # Will cause TypeError
    (100, 10, "add"),
    (20, 4, "unknown_op") # Will cause ValueError (custom handling)
]

results = []
for op_num, (val1, val2, op_type) in enumerate(operations):
    print(f"\nProcessing Operation {op_num + 1}: {val1} {op_type} {val2}")

    try:
        if op_type == "divide":
            res = val1 / val2
        elif op_type == "add":
            res = val1 + val2
        elif op_type == "multiply":
            res = val1 * val2
        else:
            raise ValueError(f"Unsupported operation type: {op_type}")
    except ZeroDivisionError:
        print(f"Error: Division by zero encountered for {val1}/{val2}. Skipping this operation.")
        res = "Error: Division by zero"
    except TypeError:
        print(f"Error: Invalid type for operation '{op_type}' with {val1}, {val2}. Skipping.")
        res = "Error: Invalid type"
    except ValueError as e:
        print(f"Error: {e}. Skipping operation.")
        res = f"Error: {e}"
    except Exception as e: # Catch any other unexpected error
        print(f"An unexpected error occurred: {e}. Skipping operation.")
        res = f"Error: Unexpected: {e}"
    
    else:
        print(f"Operation Successful. Result {res}")

    finally:
        results.append(res)
        print("--- Operation processing complete for this item ---")
    
print("\n--- Pipeline Summary ---")
for i, r in enumerate(results):
    print(f"Operation {i+1} result: {r}")



Processing Operation 1: 10 divide 2
Operation Successful. Result 5.0
--- Operation processing complete for this item ---

Processing Operation 2: 5 divide 0
Error: Division by zero encountered for 5/0. Skipping this operation.
--- Operation processing complete for this item ---

Processing Operation 3: hello multiply 2
Operation Successful. Result hellohello
--- Operation processing complete for this item ---

Processing Operation 4: 100 add 10
Operation Successful. Result 110
--- Operation processing complete for this item ---

Processing Operation 5: 20 unknown_op 4
Error: Unsupported operation type: unknown_op. Skipping operation.
--- Operation processing complete for this item ---

--- Pipeline Summary ---
Operation 1 result: 5.0
Operation 2 result: Error: Division by zero
Operation 3 result: hellohello
Operation 4 result: 110
Operation 5 result: Error: Unsupported operation type: unknown_op


In [1]:
# Handling invalind input

def get_positive_int():
    while True:
        try:
            user_input = input("Please enter a positive integer: ")
            number = int(user_input)
            if number < 0:
                print("the number must be positive. Try again")
                continue #ask again
            print(f"you entered{number}")
        except ValueError:
            print("That is not a valid integer. Try again")ç
        except KeyboardInterrupt: # Handle Ctrl + C gracefully
            print("\nOperation Canceled by user")
            return None
        else:
            print("Operation Completed successfully")
        finally:
            print("Done")
get_positive_int()

the number must be positive. Try again
Done
you entered3
Operation Completed successfully
Done
That is not a valid integer. Try again
Done
That is not a valid integer. Try again
Done
That is not a valid integer. Try again
Done
That is not a valid integer. Try again
Done
That is not a valid integer. Try again
Done
you entered45
Operation Completed successfully
Done
That is not a valid integer. Try again
Done


#### custom Exceptions

While python's in-built exceptions cover many common scenarios, sometimnes you need to define own types.
Why??
- Clarity: Provide meaninful names for errors that are specific to your applications domain or biz logic (e.g InsufficientFundsError instead of ValueError)
- Granular Handling: Calling code to catch and handle specific types of errors differently
- Abstraction: encapsulate comple error conditions               



In [27]:
# Custom Exception Hierachy

class BankError(Exception):
    """Base Error for banking related errors"""
    pass

class InsufficientFundsError(BankError):
    def __init__(self, message = "Insufficient funds for the transaction", balance = 0, amount_needed = 0):
        super().__init__(message)
        self.balance = balance
        self.amount_needed = amount_needed
    
    def __str__(self):
        return (f"{self.args[0]} Current balance: ${self.balance:,.2f}, " f"Amount needed: ${self.amount_needed:,.2f}.")


class InvalidTransactionAmountError(BankError):
    # raised when the transaction amount is Zero or Negative
    def __init__(self, message="Transaction amount must be positive", amount=0):
        super().__init__(message)
        self.amount = amount

In [23]:
def withdraw(account_balance, amount_to_withdraw):
    if amount_to_withdraw < 0:
        raise InvalidTransactionAmountError(amount = amount_to_withdraw)
    if amount_to_withdraw > account_balance:
        raise InsufficientFundsError(balance=account_balance, amount_needed=amount_to_withdraw)
    
    new_balance = account_balance - amount_to_withdraw
    print(f"Withrawal was successful. Your new balance is ${new_balance:,.2f}")
    

In [32]:
# usage with try except blocks

my_balance = 500.00

# valid withdrawal

try: 
    withdraw(my_balance, 100.00)
except BankError as e:
    print(f"Transaction Failed:{e}")

print("_____"*20)

Withrawal was successful. Your new balance is $400.00
____________________________________________________________________________________________________


In [30]:
# invalid amount

try:
    withdraw(my_balance, -90)
except InvalidTransactionAmountError as e:
    print(f"Specific error caught: {e}")
except BankError as e:
    print(f"Generic Bank error caught {e}")

Specific error caught: Transaction amount must be positive


In [34]:
print("-" * 30)

# Insufficient funds
try:
    withdraw(my_balance, 600.00)
except InsufficientFundsError as e:
    print(f"Specific error caught: {e}")
except BankError as e:
    print(f"Generic BankError caught: {e}")

------------------------------
Specific error caught: Insufficient funds for the transaction Current balance: $500.00, Amount needed: $600.00.
